In [12]:
import json
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# 1. Load, Filter, and Tidy Data
def load_and_tidy_data(filepath):
    with open(filepath, 'r') as f:
        data = json.load(f)

    rows = []
    for run in data:
        base_info = {
            'run_key': run.get('run_key'),
            'model_index': run.get('model_index'),
            'epoch': run.get('epoch')
        }
        
        for res in run.get('coherence_results', []):
            layer_name = res.get('layer', '')
            
            # CONSTRAINT 1: Only include layers starting with "lins"
            if not layer_name.startswith('lins'):
                continue
            
            # CONSTRAINT 2: Handle if "anisotropy_bounds" is present but null
            bounds = res.get('anisotropy_bounds')
            # Check if bounds is a valid dictionary before trying to access keys
            bound_0_1 = bounds.get('0.1') if isinstance(bounds, dict) else None

            layer_info = {
                'layer': layer_name,
                'subspace_coherence': res.get('subspace_coherence'),
                'subspace_dimension': res.get('subspace_dimension'),
                'full_dimension': res.get('full_dimension'),
                'mean_anisotropy_operator_norm': res.get('mean_anisotropy_operator_norm'),
                'anisotropy_bound_0_1': bound_0_1
            }
            
            # Calculate the dimension ratio safely
            full_dim = layer_info['full_dimension']
            sub_dim = layer_info['subspace_dimension']
            layer_info['dimension_ratio'] = (sub_dim / full_dim) if full_dim else None
                
            combined_info = {**base_info, **layer_info}
            
            # Explode norms for tidy data
            norms = res.get('anisotropy_operator_norms', [])
            if not norms:
                rows.append(combined_info)
            else:
                for norm in norms:
                    row = combined_info.copy()
                    row['anisotropy_operator_norm'] = norm
                    rows.append(row)

    return pd.DataFrame(rows)


# 2. Create the Dual-Axis Plot with Distinct Color Scales
def create_dual_axis_plot(df, run_key):
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    
    epochs = sorted(df['epoch'].unique())
    n_epochs = len(epochs)
    
    # CONSTRAINT 3: Distinct colors for Coherence (Blues) and Anisotropy (Reds)
    # We sample from continuous scales to ensure darker shades for later epochs
    coh_colors = px.colors.sample_colorscale("Blues", [0.4 + 0.6 * (i / max(1, n_epochs - 1)) for i in range(n_epochs)])
    ani_colors = px.colors.sample_colorscale("Reds", [0.4 + 0.6 * (i / max(1, n_epochs - 1)) for i in range(n_epochs)])
    
    for i, ep in enumerate(epochs):
        c_color = coh_colors[i]
        a_color = ani_colors[i]
        
        # Unique rows for line plots, full rows for distributions
        ep_df_unique = df[df['epoch'] == ep].drop_duplicates(subset=['layer']).sort_values('layer')
        ep_df_full = df[df['epoch'] == ep].sort_values('layer')
        
        # --- COHERENCE (Y1, Blues) ---
        fig.add_trace(go.Scatter(
            x=ep_df_unique['layer'],
            y=ep_df_unique['subspace_coherence'],
            mode='lines+markers',
            name=f'Subspace Coherence (Ep {ep})',
            line=dict(color=c_color, width=3),
            legendgroup=f"epoch_{ep}"
        ), secondary_y=False)
        
        # --- ANISOTROPY (Y2, Reds) ---
        # 1. Anisotropy Bound (Only plot if there is non-null data)
        if ep_df_unique['anisotropy_bound_0_1'].notna().any():
            fig.add_trace(go.Scatter(
                x=ep_df_unique['layer'],
                y=ep_df_unique['anisotropy_bound_0_1'],
                mode='lines',
                name=f'Anisotropy Bound 0.1 (Ep {ep})',
                line=dict(color=a_color, dash='dot', width=2),
                legendgroup=f"epoch_{ep}"
            ), secondary_y=True)

        # 2. Mean Anisotropy
        fig.add_trace(go.Scatter(
            x=ep_df_unique['layer'],
            y=ep_df_unique['mean_anisotropy_operator_norm'],
            mode='lines+markers',
            name=f'Mean Anisotropy (Ep {ep})',
            line=dict(color=a_color, dash='dash', width=2),
            marker=dict(symbol='square', size=8),
            legendgroup=f"epoch_{ep}"
        ), secondary_y=True)

        # 3. Anisotropy Distribution per neuron (Box plots)
        fig.add_trace(go.Violin(
            x=ep_df_full['layer'],
            y=ep_df_full['anisotropy_operator_norm'],
            name=f'Anisotropy Dist (Ep {ep})',
            marker_color=a_color,
            showlegend=False,
            legendgroup=f"epoch_{ep}",
            offsetgroup=str(ep) # Keeps boxes from overlapping
        ), secondary_y=True)

    # --- INDEPENDENT MARKERS ---
    min_dim_ratio = df['dimension_ratio'].min()
    fig.add_hline(
        y=min_dim_ratio, 
        secondary_y=False, 
        line_dash="dash", 
        line_color="black", 
        annotation_text=f"Min Dim Ratio: {min_dim_ratio:.3f}",
        annotation_position="bottom right"
    )

    # Formatting
    fig.update_layout(
        boxmode='group',
        title=f"Subspace Coherence & Anisotropy per Layer: <b>{run_key}</b>",
        xaxis_title="Layers",
        hovermode="x unified"
    )

    upper_limit = max(0.01, max(df['anisotropy_operator_norm'].max(), df["mean_anisotropy_operator_norm"].max(), df['anisotropy_bound_0_1'].max())*1.05)

    fig.update_yaxes(title_text="<b>Subspace Coherence</b> & Dim Ratio (Blues)", secondary_y=False)
    fig.update_yaxes(title_text="<b>Anisotropy</b> (Reds)",
                     range=[0, upper_limit],
                     secondary_y=True)
    

    return fig

# Generate and view the plot
# fig = create_dual_axis_plot(df)
# fig.show()

from plots import plot_grid

df = load_and_tidy_data("outputs/subspace-coherence-debug.json")
figs = []
for run_key in df["run_key"].unique():
    if run_key not in ["mlp_symmetry0", "mlp_symmetry1_kappa0", "mlp_symmetry1_kappa1", "mlp_symmetry3_kappa1"]: continue
    fig = create_dual_axis_plot(df[df['run_key'] == run_key], run_key)
    figs.append(fig)
plot_grid([figs[:2], figs[2:4]], 1000, height=400)

    'data': [{'legendgroup': 'epoch_100',
              'line': {…